# Etapa 1:

## Qual o problema socioeconômico que você está investigando?

O Nordeste apresenta um clima bem heterogêneo junto de uma biodiversidade ambiental vasta, tendo uma diferença grande nos regimes de precipitação, temperatura, umidade e disponibilidade hídrica. Essa variação pode afetar justamente as queimadas através de ressecamento da vegetação e do material de combustível, provocando maior intensidade nos focos de queimadas. Nesse contexto, o problema socioeconômico investigado está relacionado aos impactos que esses eventos podem provocar sobre a população e sobre as atividades econômicas da região, especialmente na agricultura, pecuária, saúde pública e conservação dos recursos naturais. Queimadas de maior intensidade podem causar perdas de áreas produtivas, degradação ambiental, aumento da emissão de poluentes atmosféricos e maior demanda por ações de combate e prevenção por parte do poder público.

## Por que ele é relevante para o Nordeste brasileiro?

Por causa da presença de extensas áreas sujeitas a períodos de estiagem e elevada variabilidade das condições meteorológicas, além da importância das atividades agropecuárias para diversos municípios da região. A combinação entre baixa precipitação, temperaturas elevadas, baixa umidade e disponibilidade de material combustível pode favorecer condições de maior risco de fogo. Dessa forma, a análise conjunta dos dados meteorológicos disponibilizados pelo Instituto Nacional de Meteorologia (INMET) e dos dados de focos de calor e Potência Radiativa do Fogo (FRP) do Instituto Nacional de Pesquisas Espaciais (INPE), entre 2020 e 2024, permite identificar padrões espaciais e temporais, períodos críticos e áreas com maior frequência ou intensidade de queimadas.

## Que tipo de decisão um gestor público poderia tomar com base nos resultados do seu modelo?

A identificação de períodos e regiões com maior probabilidade de ocorrência ou intensidade de queimadas poderia orientar a distribuição de equipes de combate a incêndios, a intensificação da fiscalização, a definição de áreas prioritárias para monitoramento e a emissão de alertas preventivos. Além disso, os resultados poderiam auxiliar no planejamento de políticas ambientais, agrícolas e de proteção civil, permitindo que os recursos públicos sejam direcionados de forma mais eficiente para os locais e períodos de maior risco.

## Hipótese principal

Menores níveis de precipitação no dia da ocorrência e períodos mais prolongados sem chuva estão associados a uma maior probabilidade de ocorrência de focos com FRP acima da mediana (target = 1), sendo esperado que a duração do período sem chuva apresente maior associação com a intensidade do foco do que a precipitação registrada apenas no dia.

A hipótese será rejeitada caso a precipitação antecedente não contribua para aumentar a probabilidade de target = 1 ou não apresente maior capacidade preditiva que a precipitação do próprio dia.

## Hipótese secundária 1

Áreas com baixa precipitação e baixa umidade apresentam maior concentração espacial de focos com FRP acima da mediana (target = 1).

Espera-se identificar maior proporção de target = 1 nessas áreas e autocorrelação espacial positiva e significativa pelo índice de Moran Global (I > 0; p < 0,05) e Moran Local/LISA.

A hipótese será rejeitada caso não seja identificada concentração significativa de focos com target = 1 em áreas mais secas.

## Hipótese secundária 2

Maiores velocidades do vento estão associadas a uma maior probabilidade de ocorrência de focos com FRP acima da mediana (target = 1), principalmente em condições de baixa umidade e baixa precipitação antecedente.

A hipótese será rejeitada caso o vento não apresente associação positiva com a probabilidade de target = 1 ou caso essa associação não seja intensificada em condições mais secas.


# Etapa 2 (Preparação dos Dados)

In [1]:
# criando seção spark

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("INPE-MLlib")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)

d:\data_science\inpe-mllib-study\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Vamos ler primeiramente os dados de ambos as fontes (inpe e inmet) e analisar os primeiros registros

In [2]:
df_inpe = (
    spark.read
    .option("header", True)
    .option('sep', ',')
    .option('inferSchema', True)
    .csv("../data/bronze/inpe/*.csv")
)

In [3]:
df_inpe.show()

+--------------------+---------+------+--------------+------------------+--------------+-----------+------------+---------+----+-------------------+-------------------+
|            DataHora| Satelite|  Pais|        Estado|         Municipio|         Bioma|DiaSemChuva|Precipitacao|RiscoFogo| FRP|           Latitude|          Longitude|
+--------------------+---------+------+--------------+------------------+--------------+-----------+------------+---------+----+-------------------+-------------------+
| 2020/01/01 00:45:28|  METOP-B|Brasil|        PARANÁ|         ITAPERUÇU|Mata Atlântica|       11.0|         3.3|      0.2|NULL|-25.187599182128906| -49.39350128173828|
|﻿2020/01/01 00:45:28|  METOP-B|Brasil|        PARANÁ| RIO BRANCO DO SUL|Mata Atlântica|       11.0|         3.2|      0.2|NULL|-25.185699462890625|-49.383201599121094|
| 2020/01/01 00:45:53|  METOP-B|Brasil|        PARANÁ|         ITAPERUÇU|Mata Atlântica|       11.0|         3.4|      0.2|NULL|-25.225500106811523| -49.38

In [5]:
df_inpe.printSchema()

root
 |-- DataHora: string (nullable = true)
 |-- Satelite: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Bioma: string (nullable = true)
 |-- DiaSemChuva: double (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- RiscoFogo: double (nullable = true)
 |-- FRP: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



Percebe-se que a coluna de DataHora veio como string, vamos tratar isso e transformar em timestamp:

DataHora: timestamp
DiaSemChuva: integer
Precipitacao: double
RiscoFogo: double
FRP (Fire Radiative Power): double
Latitude: double
Longitude: double

In [3]:
from pyspark.sql import functions as F

df_inpe = df_inpe.withColumn(
    "DataHora",
    F.coalesce(
        F.try_to_timestamp(
            F.regexp_replace(F.col("DataHora"), r"^[^\d]+", ""),
            F.lit("yyyy/MM/dd HH:mm:ss")
        ),
        F.try_to_timestamp(
            F.regexp_replace(F.col("DataHora"), r"^[^\d]+", ""),
            F.lit("yyyy-MM-dd HH:mm:ss")
        )
    )
)

In [7]:
df_inpe.printSchema()

root
 |-- DataHora: timestamp (nullable = true)
 |-- Satelite: string (nullable = true)
 |-- Pais: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- Bioma: string (nullable = true)
 |-- DiaSemChuva: double (nullable = true)
 |-- Precipitacao: double (nullable = true)
 |-- RiscoFogo: double (nullable = true)
 |-- FRP: double (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)



Com as tipagens corretas vamos analisar agora as estatísticas descritivas mais comuns

In [8]:
df_inpe.describe().show()

+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|summary| Satelite|    Pais|   Estado|      Municipio|   Bioma|       DiaSemChuva|      Precipitacao|         RiscoFogo|              FRP|           Latitude|         Longitude|
+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|  count| 26976571|26976571| 26976571|       26976571|26976507|          26478762|          26478762|          26478762|         21335115|           26976571|          26976571|
|   mean|     NULL|    NULL|     NULL|           NULL|    NULL|19.166153047487644| 0.751714091844774|-8.485602798191671|33.89365710941816|-10.182877124779395| -52.6529186162472|
| stddev|     NULL|    NULL|     NULL|           NULL|    NULL|101.65792629735161|3.6642243221536015| 95.71982

Observa-se que o inpe usa -999 para valores nulos, o que acaba prejudicando a média e o desvio-padrão das colunas que tem isso (dia sem chvua e risco fogo), vamos trocar esses valores por NULL para deixá-los mais corretos

In [4]:
df_inpe = (
    df_inpe.replace(
        -999,
        None,
        subset=['DiaSemChuva','RiscoFogo']
    )
)

In [10]:
df_inpe.describe().show()

+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|summary| Satelite|    Pais|   Estado|      Municipio|   Bioma|       DiaSemChuva|      Precipitacao|         RiscoFogo|              FRP|           Latitude|         Longitude|
+-------+---------+--------+---------+---------------+--------+------------------+------------------+------------------+-----------------+-------------------+------------------+
|  count| 26976571|26976571| 26976571|       26976571|26976507|          26252452|          26478762|          26233778|         21335115|           26976571|          26976571|
|   mean|     NULL|    NULL|     NULL|           NULL|    NULL|27.943282974100857| 0.751714091844774|0.7643107706410313|33.89365710941816|-10.182877124779395| -52.6529186162472|
| stddev|     NULL|    NULL|     NULL|           NULL|    NULL| 37.54738917154817|3.6642243221536015|0.3355730

In [11]:
df_inpe.count()

26976571

Agora com os dados aparentemente corretos vamos ver os do inmet para fazer o join espaço-temporal

O detalhe aqui dos dados do inmet é o seguinte, temos um exemplo de estrutura:

Nome: JEREMOABO
Codigo Estacao: A450
Latitude: -10.08083332
Longitude: -38.34583333
Altitude: 261
Situacao: Pane
Data Inicial: 2020-01-01
Data Final: 2024-12-31
Periodicidade da Medicao: Horaria

Data Medicao;Hora Medicao;PRECIPITACAO TOTAL, HORARIO(mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA(mB);PRESSAO ATMOSFERICA REDUZIDA NIVEL DO MAR, AUT(mB);PRESSAO ATMOSFERICA MAX.NA HORA ANT. (AUT)(mB);PRESSAO ATMOSFERICA MIN. NA HORA ANT. (AUT)(mB);RADIACAO GLOBAL(Kj/m²);TEMPERATURA DA CPU DA ESTACAO(°C);TEMPERATURA DO AR - BULBO SECO, HORARIA(°C);TEMPERATURA DO PONTO DE ORVALHO(°C);TEMPERATURA MAXIMA NA HORA ANT. (AUT)(°C);TEMPERATURA MINIMA NA HORA ANT. (AUT)(°C);TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT)(°C);TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT)(°C);TENSAO DA BATERIA DA ESTACAO(V);UMIDADE REL. MAX. NA HORA ANT. (AUT)(%);UMIDADE REL. MIN. NA HORA ANT. (AUT)(%);UMIDADE RELATIVA DO AR, HORARIA(%);VENTO, DIRECAO HORARIA (gr)(° (gr));VENTO, RAJADA MAXIMA(m/s);VENTO, VELOCIDADE HORARIA(m/s);
2020-01-01;0000;0;983,2;1013,3;983,2;982,2;-1,9;26;24,3;21,1;25,1;24,2;22,2;20,9;12,6;84;81;83;36;4,4;1,7;
2020-01-01;0100;0;983,4;1013,6;983,4;983,2;-3,5;26;23,3;20,8;24,3;23,3;21,2;20,7;12,6;86;81;86;119;4,3;2;
2020-01-01;0200;0;983;1013,2;983,5;983;-3,5;25;23;21,1;23,3;23;21,1;20,8;12,6;89;86;89;122;3,1;1,1;
2020-01-01;0300;0;982,3;1012,5;983;982,3;-3,2;25;23,1;20,3;23,1;23;21,1;20,2;12,6;89;84;84;140;2,2;,6;
2020-01-01;0400;0;981,3;1011,5;982,3;981,3;-3;25;22,6;20,1;23,1;22,6;20,3;19,7;12,5;86;82;86;334;2,1;1,2;
2020-01-01;0500;0;981,3;1011,5;981,4;981,2;-3,3;24;22,9;19,6;22,9;22,6;20,3;19,5;12,5;87;81;82;334;2,2;,7;

Para uma estrutura assim, vamos querer pegar pelo menos as informações iniciais alí e colocar no dataframe também, então vamos fazer duas leituras de dataframes, e colocar o código que está no nome do arquivo como coluna, para depois fazer o join e deixar as informações todas em um dataframe só

A primeira leitura dos dados:

In [5]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema = StructType([
    StructField("data_medicao", StringType(), True),
    StructField("hora_medicao", StringType(), True),
    StructField("precipitacao_total_horario_mm", DoubleType(), True),
    StructField("pressao_atmosferica_estacao_horaria_mb", DoubleType(), True),
    StructField("pressao_atmosferica_nivel_mar_mb", DoubleType(), True),
    StructField("pressao_atmosferica_max_hora_ant_mb", DoubleType(), True),
    StructField("pressao_atmosferica_min_hora_ant_mb", DoubleType(), True),
    StructField("radiacao_global_kj_m2", DoubleType(), True),
    StructField("temperatura_cpu_estacao_c", DoubleType(), True),
    StructField("temperatura_ar_c", DoubleType(), True),
    StructField("temperatura_ponto_orvalho_c", DoubleType(), True),
    StructField("temperatura_max_hora_ant_c", DoubleType(), True),
    StructField("temperatura_min_hora_ant_c", DoubleType(), True),
    StructField("temperatura_orvalho_max_hora_ant_c", DoubleType(), True),
    StructField("temperatura_orvalho_min_hora_ant_c", DoubleType(), True),
    StructField("tensao_bateria_estacao_v", DoubleType(), True),
    StructField("umidade_relativa_max_hora_ant_pct", DoubleType(), True),
    StructField("umidade_relativa_min_hora_ant_pct", DoubleType(), True),
    StructField("umidade_relativa_ar_pct", DoubleType(), True),
    StructField("vento_direcao_horaria_graus", DoubleType(), True),
    StructField("vento_rajada_max_ms", DoubleType(), True),
    StructField("vento_velocidade_horaria_ms", DoubleType(), True),
])


# e aqui fazemos a leitura dos dados, já colocando também o nome do arquivo e trocando o null por valores nuloes mesmo, filtrando apenas as colunas que começam com data (colunas de dados com data)

df_inmet = (
    spark.read
    .option("header", False)
    .option("sep", ";")
    .option("nullValue", "null")
    .option("locale", "pt-BR")
    .schema(schema)
    .csv("../data/bronze/inmet/*.csv")
    .withColumn("arquivo", F.input_file_name())
    .filter(
        F.col("data_medicao").rlike(r"^\d{4}-\d{2}-\d{2}$")
    )
)

In [13]:
df_inmet.show(truncate=False)

+------------+------------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+-------------------------------------------------------------------------------------------------+
|data_medicao|hora_medicao|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|temperatura_ar_c|temperatura_ponto_orvalh

Agora ler um dataframe com as informações das estações:

In [6]:
from pyspark.sql import functions as F

df_estacoes = (
    spark.read
    .format("binaryFile")
    .load("../data/bronze/inmet/*.csv")

    .withColumn("arquivo", F.input_file_name())

    .withColumn(
        "texto",
        F.decode("content", "ISO-8859-1")
    )

    .select(
        "arquivo",

        F.regexp_extract(
            "texto",
            r"Nome:\s*([^\r\n]+)",
            1
        ).alias("nome"),

        F.regexp_extract(
            "texto",
            r"Codigo Estacao:\s*([^\r\n]+)",
            1
        ).alias("codigo_estacao"),

        F.regexp_extract(
            "texto",
            r"Latitude:\s*([^\r\n]+)",
            1
        ).cast("double").alias("latitude"),

        F.regexp_extract(
            "texto",
            r"Longitude:\s*([^\r\n]+)",
            1
        ).cast("double").alias("longitude"),

        F.regexp_extract(
            "texto",
            r"Altitude:\s*([^\r\n]+)",
            1
        ).cast("double").alias("altitude"),

        F.regexp_extract(
            "texto",
            r"Situacao:\s*([^\r\n]+)",
            1
        ).alias("situacao")
    )
)

In [15]:
df_estacoes.show(truncate=False)

+-------------------------------------------------------------------------------------------------+----------------+--------------+------------+------------+--------+--------+
|arquivo                                                                                          |nome            |codigo_estacao|latitude    |longitude   |altitude|situacao|
+-------------------------------------------------------------------------------------------------+----------------+--------------+------------+------------+--------+--------+
|file:///d:/data_science/inpe-mllib-study/data/bronze/inmet/dados_A217_H_2020-01-01_2024-12-31.csv|FAROL de SANTANA|A217          |-2.27083332 |-43.62416666|9.87    |Pane    |
|file:///d:/data_science/inpe-mllib-study/data/bronze/inmet/dados_A422_H_2020-01-01_2024-12-31.csv|ABROLHOS        |A422          |-17.96305555|-38.70333333|20.93   |Pane    |
|file:///d:/data_science/inpe-mllib-study/data/bronze/inmet/dados_A344_H_2020-01-01_2024-12-31.csv|CALCANHAR       |A344

Agora com isso vamos juntar os dois dataframes pela coluna "arquivo" que é nossa chave temporária

In [7]:
df_inmet = (
    df_inmet
    .join(
        df_estacoes,
        on="arquivo",
        how="left"
    )
    .drop("arquivo")
)

In [17]:
df_inmet.show()

+------------+------------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+----------------+--------------+-----------+------------+--------+--------+
|data_medicao|hora_medicao|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|temperatura_ar_c|temperatura_ponto_orvalho_c|temperatura_max_hor

E com isso temos nosso dataframe final para o inmet na etapa bronze, vamos verificar as principais estatisticas:

In [18]:
df_inmet.printSchema()

root
 |-- data_medicao: string (nullable = true)
 |-- hora_medicao: string (nullable = true)
 |-- precipitacao_total_horario_mm: double (nullable = true)
 |-- pressao_atmosferica_estacao_horaria_mb: double (nullable = true)
 |-- pressao_atmosferica_nivel_mar_mb: double (nullable = true)
 |-- pressao_atmosferica_max_hora_ant_mb: double (nullable = true)
 |-- pressao_atmosferica_min_hora_ant_mb: double (nullable = true)
 |-- radiacao_global_kj_m2: double (nullable = true)
 |-- temperatura_cpu_estacao_c: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- temperatura_ponto_orvalho_c: double (nullable = true)
 |-- temperatura_max_hora_ant_c: double (nullable = true)
 |-- temperatura_min_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_max_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_min_hora_ant_c: double (nullable = true)
 |-- tensao_bateria_estacao_v: double (nullable = true)
 |-- umidade_relativa_max_hora_ant_pct: double (nullable 

In [19]:
df_inmet.describe().show()

+-------+------------+-----------------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+------------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+--------+--------------+------------------+-------------------+------------------+----------+
|summary|data_medicao|     hora_medicao|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|  temperatura_ar_c

Apenas a coluna de data que não está correta, vamos juntar "data" e "hora" em uma coluna DataHora para ficar igual ao outro dataframe do inpe em formato de timestamp

In [8]:
from pyspark.sql import functions as F

df_inmet = (
    df_inmet
    .withColumn(
        "DataHora",
        F.to_timestamp(
            F.concat(
                F.col("data_medicao"),
                F.col("hora_medicao")
            ),
            "yyyy-MM-ddHHmm"
        )
    )
    .drop('data_medicao', 'hora_medicao')
)

In [21]:
df_inmet.show()

+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+----------------+--------------+-----------+------------+--------+--------+-------------------+
|precipitacao_total_horario_mm|pressao_atmosferica_estacao_horaria_mb|pressao_atmosferica_nivel_mar_mb|pressao_atmosferica_max_hora_ant_mb|pressao_atmosferica_min_hora_ant_mb|radiacao_global_kj_m2|temperatura_cpu_estacao_c|temperatura_ar_c|temperatura_ponto_orvalho_c|temperatura_max_hora_ant_c|temperatura_min_hora_ant

In [22]:
df_inmet.count()

6281160

In [23]:
df_inmet.printSchema()

root
 |-- precipitacao_total_horario_mm: double (nullable = true)
 |-- pressao_atmosferica_estacao_horaria_mb: double (nullable = true)
 |-- pressao_atmosferica_nivel_mar_mb: double (nullable = true)
 |-- pressao_atmosferica_max_hora_ant_mb: double (nullable = true)
 |-- pressao_atmosferica_min_hora_ant_mb: double (nullable = true)
 |-- radiacao_global_kj_m2: double (nullable = true)
 |-- temperatura_cpu_estacao_c: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- temperatura_ponto_orvalho_c: double (nullable = true)
 |-- temperatura_max_hora_ant_c: double (nullable = true)
 |-- temperatura_min_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_max_hora_ant_c: double (nullable = true)
 |-- temperatura_orvalho_min_hora_ant_c: double (nullable = true)
 |-- tensao_bateria_estacao_v: double (nullable = true)
 |-- umidade_relativa_max_hora_ant_pct: double (nullable = true)
 |-- umidade_relativa_min_hora_ant_pct: double (nullable = true)
 |-- umidade_re

Agora vamos fazer o join temporal entre df_inpe e df_inmet por focos de queimada e estação meteorológica mais próxima
(por latitude/longitude e data), vamos usar a formula de Haversine para isso

A ideia aqui é a seguinte, normalizar por hora a hora os dados do inmet e inpe, e depois selecionar a estação mais próxima usando Haversine, porém o detalhe é que temos estações com dados 100% nulos (em pane), então vamos filtrar por estações que tem pelo menos 1 das variáveis úteis (temperatura do ar, umidade, velocidade do vento, precipitação, radiação), assim estações como a a217 que estão em pane não serão escolhidas

O fluxo é esse:

1. criar uma chave horária;
2. criar um ID único para cada foco;
3. fazer o join temporal;
4. eliminar estações totalmente sem dados naquela hora usando OR;
5. calcular Haversine;
6. escolher a estação mais próxima com row_number();
7. selecionar as colunas finais.

In [9]:
# criando uma chave horária para ambos os dataframes

df_inpe = df_inpe.withColumn(
    "hora_ref",
    F.date_trunc("hour", F.col("DataHora"))
)

df_inmet = df_inmet.withColumn(
    "hora_ref",
    F.date_trunc("hour", F.col("DataHora"))
)

In [10]:
# criar um id único para os focos para evitar problemas com latitude e longitude, além de colocar alias

df_inpe = df_inpe.withColumn(
    "id_foco",
    F.monotonically_increasing_id()
)

f = df_inpe.alias("f")
e = df_inmet.alias("e")

In [11]:
# removendo as estações que não tem pelo menos uma das informações úteis, ou seja, retirando estações que estão em pane naquele horário
# observação, estar em pane não significa não ter dados em um momento naquela hora que foi medida, por tanto não usamos a coluna "situacao" != 'Pane" por isso
# vamos filtrar as estações do inmet que são válidas antes do join para evitar problemas de shuffle no spark

variaveis_meteo = [
    "temperatura_ar_c",
    "umidade_relativa_ar_pct",
    "precipitacao_total_horario_mm",
    "vento_velocidade_horaria_ms",
    "radiacao_global_kj_m2",
]

condicao_valida = F.lit(False)

for c in variaveis_meteo:
    condicao_valida = condicao_valida | F.col(c).isNotNull()

df_inmet = df_inmet.filter(condicao_valida)

In [12]:
df_candidatos = (
    f.join(
        e,
        (
            (F.col("f.hora_ref") == F.col("e.hora_ref"))
            &
            (
                F.abs(
                    F.col("f.Latitude") -
                    F.col("e.latitude")
                ) <= 3
            )
            &
            (
                F.abs(
                    F.col("f.Longitude") -
                    F.col("e.longitude")
                ) <= 3
            )
        ),
        "inner"
    )
)

In [14]:
df_candidatos.show()

+-------------------+--------+------+-------+-------------+--------------+-----------+------------+---------+----+--------+---------+-------------------+-------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+--------------------+--------------+------------+------------+--------+--------+-------------------+-------------------+
|           DataHora|Satelite|  Pais| Estado|    Municipio|         Bioma|DiaSemChuva|Precipitacao|RiscoFogo| FRP|Latitude|Longitude| 

In [15]:
df_candidatos.count()

156260636

In [13]:
# agora calculando a distância de haversine e selecionando somente a estação mais próxima naquele horário

RAIO_TERRA_KM = 6371.0088

df_candidatos = df_candidatos.withColumn(
    "distancia_estacao_km",
    2 * F.lit(RAIO_TERRA_KM) * F.asin(
        F.sqrt(
            F.pow(
                F.sin(
                    (
                        F.radians(F.col("e.latitude"))
                        - F.radians(F.col("f.Latitude"))
                    ) / 2
                ),
                2
            )
            +
            F.cos(F.radians(F.col("f.Latitude")))
            *
            F.cos(F.radians(F.col("e.latitude")))
            *
            F.pow(
                F.sin(
                    (
                        F.radians(F.col("e.longitude"))
                        - F.radians(F.col("f.Longitude"))
                    ) / 2
                ),
                2
            )
        )
    )
)

In [40]:
df_candidatos.show(5)

+-------------------+--------+------+-----------+---------+--------+-----------+------------+---------+----+--------------+--------------+-------------------+-------+-----------------------------+--------------------------------------+--------------------------------+-----------------------------------+-----------------------------------+---------------------+-------------------------+----------------+---------------------------+--------------------------+--------------------------+----------------------------------+----------------------------------+------------------------+---------------------------------+---------------------------------+-----------------------+---------------------------+-------------------+---------------------------+-------------+--------------+------------+------------+--------+--------+-------------------+-------------------+--------------------+
|           DataHora|Satelite|  Pais|     Estado|Municipio|   Bioma|DiaSemChuva|Precipitacao|RiscoFogo| FRP|      L

In [14]:
# agora escolhemos a estação mais próxima usando window functions:

from pyspark.sql.window import Window

window_proximidade = (
    Window
    .partitionBy(F.col("f.id_foco"))
    .orderBy(F.col("distancia_estacao_km").asc())
)

df_final = (
    df_candidatos
    .withColumn(
        "rn",
        F.row_number().over(window_proximidade)
    )
    .filter(F.col("rn") == 1)
    .drop("rn")
)

In [15]:
# selecionando as colunas úteis para análise, junto da coluna de variável alvo (FRP)

df_final = df_final.select(
    F.col("f.id_foco").alias("id_foco"),
    F.col("f.DataHora").alias("data_hora_foco"),

    F.col("f.Satelite").alias("satelite"),
    F.col("f.Estado").alias("estado"),
    F.col("f.Municipio").alias("municipio"),
    F.col("f.Bioma").alias("bioma"),

    F.col("f.FRP").alias("frp"),
    F.col("f.DiaSemChuva").alias("dia_sem_chuva"),

    F.col("f.Latitude").alias("latitude_foco"),
    F.col("f.Longitude").alias("longitude_foco"),

    F.col("e.codigo_estacao"),
    F.col("e.nome").alias("nome_estacao"),

    F.col("e.latitude").alias("latitude_estacao"),
    F.col("e.longitude").alias("longitude_estacao"),

    "distancia_estacao_km",

    F.col("e.temperatura_ar_c"),
    F.col("e.umidade_relativa_ar_pct"),
    F.col('f.Precipitacao').alias('precipitacao_inpe_mm'),
    F.col("e.precipitacao_total_horario_mm").alias('precipitacao_inmet_mm'),
    F.col('e.pressao_atmosferica_nivel_mar_mb'),
    F.col("e.vento_velocidade_horaria_ms"),
    F.col("e.radiacao_global_kj_m2"),
)

E com isso chegamos no nosso df_silver, que é a junção dos dados do inpe e inmet, sendo a granularidade um foco do inmet, tendo sua respectiva estação válida mais próxima incluida:

In [16]:
df_final.show()

+-------+-------------------+---------+-------------------+------------------+--------------+----+-------------+-------------------+------------------+--------------+--------------------+----------------+-----------------+--------------------+----------------+-----------------------+--------------------+---------------------+--------------------------------+---------------------------+---------------------+
|id_foco|     data_hora_foco| satelite|             estado|         municipio|         bioma| frp|dia_sem_chuva|      latitude_foco|    longitude_foco|codigo_estacao|        nome_estacao|latitude_estacao|longitude_estacao|distancia_estacao_km|temperatura_ar_c|umidade_relativa_ar_pct|precipitacao_inpe_mm|precipitacao_inmet_mm|pressao_atmosferica_nivel_mar_mb|vento_velocidade_horaria_ms|radiacao_global_kj_m2|
+-------+-------------------+---------+-------------------+------------------+--------------+----+-------------+-------------------+------------------+--------------+------------

In [17]:
df_final.printSchema()

root
 |-- id_foco: long (nullable = false)
 |-- data_hora_foco: timestamp (nullable = true)
 |-- satelite: string (nullable = true)
 |-- estado: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- bioma: string (nullable = true)
 |-- frp: double (nullable = true)
 |-- dia_sem_chuva: double (nullable = true)
 |-- latitude_foco: double (nullable = true)
 |-- longitude_foco: double (nullable = true)
 |-- codigo_estacao: string (nullable = true)
 |-- nome_estacao: string (nullable = true)
 |-- latitude_estacao: double (nullable = true)
 |-- longitude_estacao: double (nullable = true)
 |-- distancia_estacao_km: double (nullable = true)
 |-- temperatura_ar_c: double (nullable = true)
 |-- umidade_relativa_ar_pct: double (nullable = true)
 |-- precipitacao_inpe_mm: double (nullable = true)
 |-- precipitacao_inmet_mm: double (nullable = true)
 |-- pressao_atmosferica_nivel_mar_mb: double (nullable = true)
 |-- vento_velocidade_horaria_ms: double (nullable = true)
 |-- radiac

Com isso agora basta salvar os dados em bronze/inpe_inmet para continuar a análise, com isso vamos particionar por ano caso seja útil mais pra frente

In [18]:
df_final = df_final.withColumn(
    "ano",
    F.year("data_hora_foco")
)

(
    df_final.write
    .mode("overwrite")
    .partitionBy("ano")
    .parquet("../data/bronze/inpe_inmet")
)

baixar os dados:
inpe -> baixe de https://data.inpe.br/queimadas/bdqueimadas/#exportar-dados, selecione apenas estados do nordeste e periodo 2020 a 2024, baixe um zip por ano, depois extraia os csvs e coloque na pasta data/bronze/

inmet -> baixe de https://bdmep.inmet.gov.br/, siga o passo a passo, selecione virgula, dados horários, estações automáticas, selecione região

diario de bordo:

- tive dificuldade com pensar em juntar os dois dados, levei 2 dias pra fazer isso
- ao terminar de conseguir executar, o df_silver deu outofmemoryerror por conta do tamanho do dataset gigantesco que acabou explodindo, e tive que analisar estratégias para contornar isso, acabei aumentando os cores do spark para usar todos e aloquei 8gb no driver memory para conseguir resolver, antes era 2gb